<a href="https://colab.research.google.com/github/beyzahiz/Cats-Dogs-CNN-Classifier/blob/main/Cats_Dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/cats_dogs_dataset"

train_dir = DATA_PATH + "/training_set"
test_dir = DATA_PATH + "/test_set"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import tensorflow as tf
#tf.keras.utils.get_file() internetten dosya indirir

def get_datasets(data_dir):
    train_ds = tf.keras.utils.image_dataset_from_directory( #görselleri yükler, label atar, batch oluşturur
        data_dir,
        validation_split= 0.2,
        subset="training",
        seed=42,
        image_size=(150,150),
        batch_size=32
    )

    val_ds=tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=(150,150),
        batch_size=32
    )

    return train_ds, val_ds

In [49]:
#Data Augmentation
#Eldeki verileri hafifçe değiştirerek daha fazla veri varmış gibi çeşitlilik oluşturmak
#Data augmentation sadece training dataset’te olmalı.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"), #görseli yatayda rastgele çevirir
    layers.RandomRotation(0.1), #%10 oranında döndürür
    layers.RandomZoom(0.1), #Hafif yakınlaştırma
])


In [50]:
IMG_SIZE=(150,150)

model=keras.Sequential([
    layers.Input(shape=(150,150,3)),
    data_augmentation,
    layers.Rescaling(1./255), #normalizasyon

    #Conv
    #resmin üzerindeki featuresları yakalamaya yarar. resmin içindeki kenarları, köşeleri ve desenleri bulur.
    layers.Conv2D(32, (3,3), activation="relu"), #32 filtre sayısı, 3,3 kernel sayısı
    layers.MaxPooling2D(2,2), #resmi özetler, boyutu küçülür ama en önemli özellikler (mesela kedinin kulağının sivri ucu) korunur.

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"), #128 nöronla karmaşık kuralları öğrenir
    layers.Dropout(0.5), #modelin ezberlemesini engeller, sadece belirli nöronlara güvenmek yerine bilgiyi tüm ağa yaymayı öğrenir.
    layers.Dense(1, activation="sigmoid") #1 çıktı üretilir ve olasılığa dönüşür
])

In [51]:
model.summary()




Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_17 (Sequential)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_8 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [52]:
#binary_crossentropy uygulayacağım çünkü 2 sınıf var çıktı 1 tane sigmoid kullanıyorum
#categorical_crossentropy +3 sınıf varsa softmax kullanılıyorsa

train_ds, val_ds = get_datasets(train_dir)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

#Early Stopping
#model eğitimi aşamasında kullanılan callback fonksiyonu. overfitting önler
early_stop=keras.callbacks.EarlyStopping(
    monitor="val_loss", #modelin hangi değeri takip edeceği
    patience=5, #eğer val_loss değeri 5 epoch boyunca iyileşmezse eğitimi durdurur.
    restore_best_weights=True #eğitimi durdurduğunda, modelin en iyi performansı gösterdiği (hatanın en düşük olduğu) ana geri döner
)

history= model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

Found 8005 files belonging to 2 classes.
Using 6404 files for training.
Found 8005 files belonging to 2 classes.
Using 1601 files for validation.
Epoch 1/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 61s 291ms/step - accuracy: 0.5220 - loss: 0.7109 - val_accuracy: 0.5921 - val_loss: 0.6492
Epoch 2/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 27s 136ms/step - accuracy: 0.6022 - loss: 0.6623 - val_accuracy: 0.5953 - val_loss: 0.6556
Epoch 3/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 27s 136ms/step - accuracy: 0.6297 - loss: 0.6455 - val_accuracy: 0.6727 - val_loss: 0.5936
Epoch 4/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 28s 137ms/step - accuracy: 0.6614 - loss: 0.6215 - val_accuracy: 0.6852 - val_loss: 0.5997
Epoch 5/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 27s 135ms/step - accuracy: 0.6704 - loss: 0.6096 - val_accuracy: 0.6952 - val_loss: 0.5906
Epoch 6/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 27s 135ms/step - accuracy: 0.6645 - loss: 0.6009 - val_accuracy: 0.7252 - val_loss: 0.5534
Epoch 7/20
201/201 ━━━━━━━━━━━━━━━━━━━━ 28s 137ms/step - accuracy: 0